In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier, Pool
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

# pd.set_option('display.max_columns', None)

In [ ]:
# loading data
train_dr = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_dr = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
train_dr

In [ ]:
# reproducibility package
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
train_dr.info()
print('\n'f'Missing values: {train_dr.isna().sum().sum()}')

# CatBoost baseline

In [ ]:
X = train_dr.drop(['label'], axis=1)
y = train_dr.label

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y)

In [ ]:
# create a sparse matrix
Xtr = csr_matrix(X_train.astype('float32') / 255.0)
Xvl = csr_matrix(X_val.astype('float32') / 255.0)
train_pool = Pool(Xtr, label=y_train)
val_pool = Pool(Xvl, label=y_val)
test_dr_pool = Pool(test_dr)

In [ ]:
clf = CatBoostClassifier(iterations=2000,
                         loss_function='MultiClass',
                         eval_metric='Accuracy',
                         task_type='GPU',
                         random_state=SEED)

clf.fit(
    train_pool,
    eval_set=val_pool,
    verbose=50,
    use_best_model=True,
    early_stopping_rounds=150
)

**score 0.96328**

# MLP

## Default Model

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# train data preparation
X = train_dr.drop('label', axis=1).values.astype('float32') / 255.0
y = train_dr.label.values.astype('int64')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y)

X_train_tz = torch.from_numpy(X_train)
X_val_tz = torch.from_numpy(X_val)
y_train_tz = torch.from_numpy(y_train)
y_val_tz = torch.from_numpy(y_val)

train_dataset = TensorDataset(X_train_tz, y_train_tz)
val_dataset = TensorDataset(X_val_tz, y_val_tz)

g = torch.Generator()
g.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
# test data preparation
X_test = test_dr.values.astype('float32') / 255.0
X_test_tz = torch.from_numpy(X_test)
test_loader = DataLoader(TensorDataset(X_test_tz), batch_size=256, shuffle=False)

In [ ]:
# create model
class MLP_Default(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP_Default().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_func = nn.CrossEntropyLoss()

In [ ]:
best_accuracy = 0.0
best_state = None

for epoch in range(30):
    # train
    model.train()
    for X_tr, y_tr in train_loader:
        X_tr, y_tr = X_tr.to(device), y_tr.to(device)
        optimizer.zero_grad()
        loss = loss_func(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()

    # eval
    model.eval()
    correct = total_obj = 0
    with torch.no_grad():
        for X_vl, y_vl in val_loader:
            X_vl, y_vl = X_vl.to(device), y_vl.to(device)
            pred = model(X_vl).argmax(1)
            correct += (pred == y_vl).sum().item()
            total_obj += y_vl.size(0)

    val_accuracy = correct / total_obj
    print(f"epoch {epoch+1}: val_accuracy={val_accuracy:.4f}")

    # save best parameters
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_state = model.state_dict()
        print(f"New best! Saving model with accuracy={best_accuracy:.4f}")

# load the best weights
model.load_state_dict(best_state)

**score 0.97703**

## Promoted Model

*I will train the model using augmentations, and I have also improved it by adding dropout, BatchNorm1d, a scheduler, and so on. Predictions will be made using the TTA method with averaging over multiple predictions. The final prediction will be obtained by averaging the predictions across different seeds.*

In [ ]:
from torch.utils.data import Dataset
import torchvision.transforms as T

In [ ]:
# data preparation with augmentation
X = train_dr.drop('label', axis=1).values.astype('uint8')
y = train_dr.label.values.astype('int64')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y
)

# augmentation
train_transform = T.Compose([
    T.ToPILImage(),    # (28 x 28) dtype=PIL.Image [0..255]
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.95, 1.05), shear=5),
    T.ToTensor(),    # (1, 28 x 28) dtype=torch.float32 [0..1]
])

eval_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
])

# class for online augmentation
class FlatAugDataset(Dataset):
    def __init__(self, X, y=None, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].reshape(28, 28)    # (784) dtype=uint8 [0..255]  ->  (28 x 28) dtype=uint8 [0..255]
        x = self.transform(x)    # tensor (1, 28 x 28) dtype=torch.float32 [0..1]
        x_vec = x.view(-1)    # (784) dtype=torch.float32 [0..1]
        if self.y is None:
            return x_vec
        return x_vec, torch.tensor(self.y[idx], dtype=torch.long)

# Datasets
train_dataset = FlatAugDataset(X_train, y_train, transform=train_transform)
val_dataset = FlatAugDataset(X_val, y_val, transform=eval_transform)

X_test = test_dr.values.astype('uint8')
test_dataset = FlatAugDataset(X_test, y=None, transform=eval_transform)

# DataLoaders
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
class MLP_Maxed(nn.Module):
    def __init__(self):
        super().__init__()
        self.fe = nn.Sequential(
            nn.Linear(784, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.head = nn.Linear(256, 10)

        # kaiming init for Linear - under ReLU/GELU
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.fe(x)
        return self.head(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP_Maxed().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
epochs = 35
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
loss_func = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
best_accuracy = 0.0
best_state = None

for epoch in range(epochs):
    # train
    model.train()
    for X_tr, y_tr in train_loader:
        X_tr, y_tr = X_tr.to(device), y_tr.to(device)
        optimizer.zero_grad()
        loss = loss_func(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()
    scheduler.step()
    
    # eval
    model.eval()
    correct = total_obj = 0
    with torch.no_grad():
        for X_vl, y_vl in val_loader:
            X_vl, y_vl = X_vl.to(device), y_vl.to(device)
            pred = model(X_vl).argmax(1)
            correct += (pred == y_vl).sum().item()
            total_obj += y_vl.size(0)

    val_accuracy = correct / total_obj
    print(f"epoch {epoch+1}: val_accuracy={val_accuracy:.4f}")

    # save best parameters
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_state = model.state_dict()
        print(f"New best! Saving model with accuracy={best_accuracy:.4f}")

# load the best weights
model.load_state_dict(best_state)

In [ ]:
# TTA transformer set
tta_transforms = [
    eval_transform,    # baseline without augmentations
    T.Compose([T.ToPILImage(), T.RandomAffine(degrees=8, translate=(0.08, 0.08)), T.ToTensor()]),
    T.Compose([T.ToPILImage(), T.RandomAffine(degrees=0, translate=(0.10, 0.00), scale=(0.98, 1.02)), T.ToTensor()]),
    T.Compose([T.ToPILImage(), T.RandomAffine(degrees=10, translate=(0.00, 0.08), shear=5), T.ToTensor()]),
]

def collect_logits(model, loader, device):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for X in loader:              
            X = X.to(device)
            all_logits.append(model(X))    # (B=256, C=10)
    return torch.cat(all_logits, dim=0) # (N=28000, C)

# accumulate logits by transformations
sum_logits = None
for tf in tta_transforms:
    test_dataset.transform = tf    # replace transform in the existing dataset
    logits = collect_logits(model, test_loader, device)  # (N, C)
    sum_logits = logits if sum_logits is None else (sum_logits + logits)

# final marks
preds = sum_logits.argmax(dim=1).cpu().numpy()

# submission
submission = pd.DataFrame({'ImageId': np.arange(1, len(preds)+1), 'Label': preds})
submission.to_csv('submission.csv', index=False)